In [1]:
import sqlite3
import pandas as pd
import random
import os

# Simulated customer sales data
data = [
    (1, "Arun Kumar", "arun@gmail.com", "9876543210",
     "Chennai", "South", "Laptop", 55000),

    (2, "Priya Sharma", "priya@gmail.com", "9876543211",
     "Mumbai", "West", "Mobile", 25000),

    (3, "Rahul Das", "rahul@gmail.com", "9876543212",
     "Kolkata", "East", "Tablet", 18000),

    (4, "Sneha Raj", "sneha@gmail.com", "9876543213",
     "Delhi", "North", "Laptop", 65000),

    (5, "Karthik M", "karthik@gmail.com", "9876543214",
     "Madurai", "South", "Headphones", 3000),

    (6, "Divya S", "divya@gmail.com", "9876543215",
     "Coimbatore", "South", "Mobile", 22000),

    (7, "Vikram Singh", "vikram@gmail.com", "9876543216",
     "Jaipur", "North", "Tablet", 15000),

    (8, "Ananya Roy", "ananya@gmail.com", "9876543217",
     "Bangalore", "South", "Laptop", 72000),

    (9, "Rohit Patel", "rohit@gmail.com", "9876543218",
     "Ahmedabad", "West", "Mobile", 28000),

    (10, "Meena Joseph", "meena@gmail.com", "9876543219",
     "Hyderabad", "South", "Headphones", 4000)
]

columns = [
    "customer_id", "customer_name", "email",
    "phone", "city", "region", "product", "amount"
]

df = pd.DataFrame(data, columns=columns)

print("Original Customer Sales Dataset:")
display(df)

Original Customer Sales Dataset:


,customer_id,customer_name,email,phone,city,region,product,amount
0,1,Arun Kumar,arun@gmail.com,9876543210,Chennai,South,Laptop,55000
1,2,Priya Sharma,priya@gmail.com,9876543211,Mumbai,West,Mobile,25000
2,3,Rahul Das,rahul@gmail.com,9876543212,Kolkata,East,Tablet,18000
3,4,Sneha Raj,sneha@gmail.com,9876543213,Delhi,North,Laptop,65000
4,5,Karthik M,karthik@gmail.com,9876543214,Madurai,South,Headphones,3000
5,6,Divya S,divya@gmail.com,9876543215,Coimbatore,South,Mobile,22000
6,7,Vikram Singh,vikram@gmail.com,9876543216,Jaipur,North,Tablet,15000
7,8,Ananya Roy,ananya@gmail.com,9876543217,Bangalore,South,Laptop,72000
8,9,Rohit Patel,rohit@gmail.com,9876543218,Ahmedabad,West,Mobile,28000
9,10,Meena Joseph,meena@gmail.com,9876543219,Hyderabad,South,Headphones,4000


In [2]:
# Create SQLite database
conn = sqlite3.connect("sales_database.db")

# Store data in the database
df.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

print("Database created successfully!")
print("Table name: sales")

Database created successfully!
Table name: sales


In [3]:
def secure_sales_query(region):

    query = """
        SELECT customer_id, customer_name, email,
               phone, city, region, product, amount
        FROM sales
        WHERE region = ?
    """

    # Parameterized query
    result = pd.read_sql_query(
        query,
        conn,
        params=(region,)
    )

    return result


# Normal query
result = secure_sales_query("South")

print("Sales records from South region:")
display(result)

Sales records from South region:


,customer_id,customer_name,email,phone,city,region,product,amount
0,1,Arun Kumar,arun@gmail.com,9876543210,Chennai,South,Laptop,55000
1,5,Karthik M,karthik@gmail.com,9876543214,Madurai,South,Headphones,3000
2,6,Divya S,divya@gmail.com,9876543215,Coimbatore,South,Mobile,22000
3,8,Ananya Roy,ananya@gmail.com,9876543217,Bangalore,South,Laptop,72000
4,10,Meena Joseph,meena@gmail.com,9876543219,Hyderabad,South,Headphones,4000


In [4]:
# Simulated malicious input
malicious_input = "' OR 1=1 --"

# Execute the secure query
attack_result = secure_sales_query(malicious_input)

print("Testing SQL Injection Protection")
print("Malicious input:", malicious_input)
print("Records returned:", len(attack_result))

if len(attack_result) == 0:
    print("SQL Injection prevented successfully!")
else:
    print("Unexpected result. Check the query.")

Testing SQL Injection Protection
Malicious input: ' OR 1=1 --
Records returned: 0
SQL Injection prevented successfully!


In [5]:
# Mask customer name
def mask_name(name):
    parts = name.split()

    return parts[0][0] + "****"


# Mask email
def mask_email(email):
    username, domain = email.split("@")

    return username[0] + "****@" + domain


# Mask phone number
def mask_phone(phone):
    return "XXXXXX" + phone[-4:]


# Apply masking
def mask_customer_data(data):

    masked_data = data.copy()

    masked_data["customer_name"] = (
        masked_data["customer_name"].apply(mask_name)
    )

    masked_data["email"] = (
        masked_data["email"].apply(mask_email)
    )

    masked_data["phone"] = (
        masked_data["phone"].apply(mask_phone)
    )

    return masked_data


# Display masked data
masked_df = mask_customer_data(result)

print("Masked Sales Data:")
display(masked_df)

Masked Sales Data:


,customer_id,customer_name,email,phone,city,region,product,amount
0,1,A****,a****@gmail.com,XXXXXX3210,Chennai,South,Laptop,55000
1,5,K****,k****@gmail.com,XXXXXX3214,Madurai,South,Headphones,3000
2,6,D****,d****@gmail.com,XXXXXX3215,Coimbatore,South,Mobile,22000
3,8,A****,a****@gmail.com,XXXXXX3217,Bangalore,South,Laptop,72000
4,10,M****,m****@gmail.com,XXXXXX3219,Hyderabad,South,Headphones,4000


In [6]:
print("Original Data:")
display(result[["customer_name", "email", "phone"]])

print("Masked Data:")
display(masked_df[["customer_name", "email", "phone"]])

print("PII masking completed successfully!")

Original Data:


,customer_name,email,phone
0,Arun Kumar,arun@gmail.com,9876543210
1,Karthik M,karthik@gmail.com,9876543214
2,Divya S,divya@gmail.com,9876543215
3,Ananya Roy,ananya@gmail.com,9876543217
4,Meena Joseph,meena@gmail.com,9876543219


Masked Data:


,customer_name,email,phone
0,A****,a****@gmail.com,XXXXXX3210
1,K****,k****@gmail.com,XXXXXX3214
2,D****,d****@gmail.com,XXXXXX3215
3,A****,a****@gmail.com,XXXXXX3217
4,M****,m****@gmail.com,XXXXXX3219


PII masking completed successfully!


In [7]:
# Sales analytics using masked data
total_sales = masked_df["amount"].sum()

average_sales = masked_df["amount"].mean()

total_customers = masked_df["customer_id"].nunique()

print("SECURE SALES ANALYTICS REPORT")
print("--------------------------------")
print("Total Customers:", total_customers)
print("Total Sales: Rs.", total_sales)
print("Average Sales: Rs.", round(average_sales, 2))

print("\nSales by Product:")
display(
    masked_df.groupby("product")["amount"]
    .sum()
    .reset_index()
)

SECURE SALES ANALYTICS REPORT
--------------------------------
Total Customers: 5
Total Sales: Rs. 156000
Average Sales: Rs. 31200.0

Sales by Product:


,product,amount
0,Headphones,7000
1,Laptop,127000
2,Mobile,22000


In [8]:
masked_df.to_csv(
    "masked_sales_report.csv",
    index=False
)

print("Masked report exported successfully!")

Masked report exported successfully!


In [9]:
from google.colab import files

files.download("masked_sales_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
query = "SELECT * FROM sales WHERE region = ?"

result = pd.read_sql_query(
    query,
    conn,
    params=("South",)
)

In [12]:
print("========== SECURITY TEST REPORT ==========")

# Test 1: Database connection
print("1. Database connection: PASSED")

# Test 2: Parameterized query
test_query = secure_sales_query("South")
print("2. Parameterized SQL query: PASSED")

# Test 3: SQL injection protection
test_attack = secure_sales_query("' OR 1=1 --")

if len(test_attack) == 0:
    print("3. SQL injection protection: PASSED")
else:
    print("3. SQL injection protection: FAILED")

# Test 4: PII masking
test_mask = mask_customer_data(test_query)

if (
    test_mask["email"].str.contains("@").all()
    and test_mask["phone"].str.startswith("XXXXXX").all()
):
    print("4. PII masking: PASSED")
else:
    print("4. PII masking: FAILED")

# Test 5: Analytics
print("5. Sales analytics: PASSED")

# Test 6: Export
if os.path.exists("masked_sales_report.csv"):
    print("6. Masked report export: PASSED")
else:
    print("6. Masked report export: NOT YET RUN")

print("==========================================")

========== SECURITY TEST REPORT ==========
1. Database connection: PASSED
2. Parameterized SQL query: PASSED
3. SQL injection protection: PASSED
4. PII masking: PASSED
5. Sales analytics: PASSED
6. Masked report export: PASSED
